# Amazon SageMaker: Business Entity Resolution Pipeline
### Scalable Inverted Index Blocking, Vectorized Feature Extraction & Macro $F_{0.5}$ GBDT Matcher

---

### Overview
This notebook provides the complete workflow to execute the Business Entity Resolution pipeline on **AWS SageMaker**.
It supports both:
1. **Direct Studio / Notebook Execution**: Run directly in a SageMaker Studio notebook or on a single `ml.m5.2xlarge` / `ml.c5.4xlarge` instance.
2. **SageMaker Managed Training & Processing Jobs**: Launch decoupled, serverless spot/on-demand compute instances running `sagemaker_train_and_inference.py`.

---

## 1. Install & Import Dependencies

In [ ]:
# Install pure-algorithm packages and AWS SDK
!pip install -q rapidfuzz lightgbm scikit-learn pandas numpy tqdm boto3 sagemaker

import os
import sys
import json
import time
import shutil
import zipfile
import boto3
import sagemaker
from sagemaker import get_execution_role

print("Dependencies loaded successfully.")

## 2. Configure AWS Environment, IAM Role & S3 Bucket

In [ ]:
# Initialize SageMaker session and default S3 bucket
try:
    role = get_execution_role()
    sess = sagemaker.Session()
    bucket = sess.default_bucket()
    region = sess.boto_region_name
    print(f"AWS SageMaker Session Initialized:")
    print(f"  Region:  {region}")
    print(f"  Bucket:  {bucket}")
    print(f"  IAM Role: {role}")
except Exception as e:
    print(f"Local environment / running outside AWS SageMaker Studio: {e}")
    role = None
    bucket = None

# Prefix for entity resolution pipeline data in S3
S3_PREFIX = "amazon-ml-challenge-2026/entity-resolution"
LOCAL_DATA_DIR = "dataset"
OUTPUT_DIR = "output"
MODEL_DIR = "models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## 3. S3 Data Synchronization
Sync the training and test dataset between local filesystem and AWS S3.

In [ ]:
# Upload local dataset to S3 (if running in SageMaker Studio with local dataset)
if bucket and os.path.exists(LOCAL_DATA_DIR):
    print(f"Uploading local dataset to s3://{bucket}/{S3_PREFIX}/dataset ...")
    sess.upload_data(path=LOCAL_DATA_DIR, bucket=bucket, key_prefix=f"{S3_PREFIX}/dataset")
    print("S3 Upload Complete!")
elif bucket:
    print(f"Download dataset from S3 if present:")
    print(f"  aws s3 sync s3://{bucket}/{S3_PREFIX}/dataset {LOCAL_DATA_DIR}/")

## 4. Run End-to-End Pipeline in SageMaker Studio / Notebook Instance
Executes training, threshold optimization, and vectorized inference on all 1,732,544 test entities.

In [ ]:
# Run the production entrypoint script sagemaker_train_and_inference.py
t0 = time.time()
!python sagemaker_train_and_inference.py \
    --version 1 \
    --train {LOCAL_DATA_DIR} \
    --test {LOCAL_DATA_DIR} \
    --model-dir {MODEL_DIR} \
    --output-data-dir {OUTPUT_DIR} \
    --train-samples 50000 \
    --mode all

print(f"Local pipeline execution completed in {time.time() - t0:.2f} seconds.")

## 5. Launch Scalable AWS SageMaker Training Job (Managed Compute Mode)
Launches an isolated SageMaker managed job on `ml.c5.4xlarge` or `ml.m5.2xlarge` compute.

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

if role and bucket:
    print("Configuring SageMaker SKLearn Estimator for remote managed execution...")
    estimator = SKLearn(
        entry_point="sagemaker_train_and_inference.py",
        source_dir=".",
        role=role,
        instance_count=1,
        instance_type="ml.c5.4xlarge",  # 16 vCPUs, 32 GiB RAM for fast inference
        framework_version="1.2-1",
        py_version="py3",
        hyperparameters={
            "version": 1,
            "train-samples": 50000,
            "mode": "all",
        },
        use_spot_instances=True,  # Cost-effective spot instances (saves up to 70%)
        max_wait=14400,
        max_run=10800,
    )
    print("To launch the job, uncomment and run: estimator.fit({'train': f's3://{bucket}/{S3_PREFIX}/dataset/train', 'test': f's3://{bucket}/{S3_PREFIX}/dataset/test'})")
else:
    print("Skipping managed job config: No active AWS credentials or IAM role.")

## 6. Official Submission Validator Check
Verifies formatting, exact row counts (1,732,544 rows), column headers, and ID constraints.

In [ ]:
validator_script = "student_resource/utils/validate_submission.py"
if not os.path.exists(validator_script):
    validator_script = "utils/validate_submission.py"

test_dir = os.path.join(LOCAL_DATA_DIR, "test") if os.path.exists(os.path.join(LOCAL_DATA_DIR, "test")) else LOCAL_DATA_DIR

!python {validator_script} \
    --matching {OUTPUT_DIR}/matching_results.tsv \
    --candidate {OUTPUT_DIR}/candidate_pairs.tsv \
    --test-dir {test_dir}

## 7. Package Final Submission Zip & Sync to S3
Bundles outputs, source code, and methodology documentation, and uploads to Amazon S3.

In [ ]:
# Create submission.zip
ZIP_PATH = "submission.zip"
print(f"Creating {ZIP_PATH}...")

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    # 1. Output TSVs and CSV
    zf.write(f"{OUTPUT_DIR}/matching_results.tsv", "output/matching_results.tsv")
    zf.write(f"{OUTPUT_DIR}/candidate_pairs.tsv", "output/candidate_pairs.tsv")
    csv_path = f"{OUTPUT_DIR}/matching_pairs.csv"
    if os.path.exists(csv_path):
        zf.write(csv_path, "output/matching_pairs.csv")
    print("  + Added output files")

    # 2. Documentation
    doc_path = "Documentation_template.md"
    if not os.path.exists(doc_path):
        doc_path = "student_resource/Documentation_template.md"
    if os.path.exists(doc_path):
        zf.write(doc_path, "Documentation_template.md")
        print("  + Added Documentation_template.md")

    # 3. Source code
    code_src_dir = "src"
    for root, dirs, files in os.walk(code_src_dir):
        for f in files:
            if not f.endswith(".pyc"):
                full_p = os.path.join(root, f)
                zf.write(full_p, os.path.join("code/business_entity_resolution", full_p))
    zf.write("sagemaker_train_and_inference.py", "code/business_entity_resolution/run_pipeline.py")
    zf.write("requirements.txt", "code/business_entity_resolution/requirements.txt")
    zf.write("README.md", "code/business_entity_resolution/README.md")
    print("  + Added source code under code/business_entity_resolution/")

print(f"\n[SUCCESS] Built {ZIP_PATH} ({os.path.getsize(ZIP_PATH):,} bytes).")

# Sync to S3 bucket
if bucket:
    s3_dest = f"s3://{bucket}/{S3_PREFIX}/submissions/{ZIP_PATH}"
    print(f"Syncing submission to S3: {s3_dest} ...")
    sess.upload_data(path=ZIP_PATH, bucket=bucket, key_prefix=f"{S3_PREFIX}/submissions")
    print(f"[OK] Uploaded to S3 successfully!")